In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install ultralytics -q

In [ ]:
import os
import ast
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
DATA_DIR = "/kaggle/input/competitions/global-wheat-detection"

TRAIN_IMG_DIR = f"{DATA_DIR}/train"

df = pd.read_csv(f"{DATA_DIR}/train.csv")

df.head()

In [ ]:
df["bbox"] = df["bbox"].apply(ast.literal_eval)

df[["x", "y", "w", "h"]] = pd.DataFrame(df["bbox"].tolist(), index=df.index)

df.head()

In [ ]:
df["x_center"] = (df["x"] + df["w"] / 2) / df["width"]
df["y_center"] = (df["y"] + df["h"] / 2) / df["height"]
df["w_norm"] = df["w"] / df["width"]
df["h_norm"] = df["h"] / df["height"]

df.head()

In [ ]:
image_ids = df["image_id"].unique()

train_ids, valid_ids = train_test_split(
    image_ids,
    test_size=0.2,
    random_state=42
)

len(train_ids), len(valid_ids)

In [ ]:
BASE_DIR = "/kaggle/working/wheat_yolo"

os.makedirs(f"{BASE_DIR}/images/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/images/valid", exist_ok=True)
os.makedirs(f"{BASE_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/labels/valid", exist_ok=True)

In [ ]:
import shutil

def make_yolo_dataset(image_ids, split):
    for image_id in tqdm(image_ids):
        src_img = f"{TRAIN_IMG_DIR}/{image_id}.jpg"
        dst_img = f"{BASE_DIR}/images/{split}/{image_id}.jpg"
        shutil.copy(src_img, dst_img)
        
        boxes = df[df["image_id"] == image_id]
        
        label_path = f"{BASE_DIR}/labels/{split}/{image_id}.txt"
        
        with open(label_path, "w") as f:
            for _, row in boxes.iterrows():
                class_id = 0
                x_center = row["x_center"]
                y_center = row["y_center"]
                w = row["w_norm"]
                h = row["h_norm"]
                
                f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

In [ ]:
yaml_content = f"""
path: {BASE_DIR}
train: images/train
val: images/valid

names:
  0: wheat
"""

with open(f"{BASE_DIR}/data.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

In [ ]:
import os

print("BASE_DIR:", BASE_DIR)
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)

print("원본 train 이미지 일부:")
print(os.listdir(TRAIN_IMG_DIR)[:5])

print("YOLO train 이미지 개수:")
print(len(os.listdir(f"{BASE_DIR}/images/train")))

print("YOLO valid 이미지 개수:")
print(len(os.listdir(f"{BASE_DIR}/images/valid")))

In [ ]:
make_yolo_dataset(train_ids, "train")
make_yolo_dataset(valid_ids, "valid")

In [ ]:
print(len(os.listdir(f"{BASE_DIR}/images/train")))
print(len(os.listdir(f"{BASE_DIR}/images/valid")))

In [ ]:
model.train(
    data=f"{BASE_DIR}/data.yaml",
    epochs=100,
    patience=20,
    imgsz=1024,
    multi_scale=True,
    batch=8,
    name="wheat_freeze10",

    fliplr=0.5,
    flipud=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    scale=0.5,
    translate=0.1,
    mosaic=1.0,

    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,

    freeze=10
)

In [ ]:
import glob

glob.glob("/kaggle/working/runs/detect/*/weights/best.pt")

In [ ]:
best_model = YOLO(
    "/kaggle/working/runs/detect/wheat_img1024/weights/best.pt"
)

results = best_model.predict(
    source=f"{BASE_DIR}/images/valid",
    conf=0.25,
    save=True
)

In [ ]:
metrics = best_model.val(
    data=f"{BASE_DIR}/data.yaml",
    split="val"
)

In [ ]:
print(metrics.box.map)      # mAP50-95
print(metrics.box.map50)    # mAP@0.5
print(metrics.box.map75)    # mAP@0.75
print(metrics.box.mp)       # precision
print(metrics.box.mr)       # recall